# 08 — Big Island DEM Mosaic

The original `kona_DEM_utm_full.tif` covers only one 1° USGS tile (19–20°N,
155–156°W), leaving the eastern coast (Hilo/Puna), southern tip (Ka Lae), and
northern tip (North Kohala) uncovered.

This notebook downloads the three missing USGS 3DEP 1/3 arc-second tiles and
merges all four into a single 500m-resolution island-wide mosaic.

| Tile | Coverage | Source |
|------|----------|--------|
| `kona_DEM_utm_full.tif` | 19–20°N, 155–156°W | already in repo (UTM 32604) |
| `USGS_13_n20w155_20250611.tif` | 19–20°N, 154–155°W | USGS TNM (Hilo/Puna coast) |
| `USGS_13_n19w156_20230522.tif` | 18–19°N, 155–156°W | USGS TNM (Ka'u / Ka Lae) |
| `USGS_13_n21w156_20230522.tif` | 20–21°N, 155–156°W | USGS TNM (North Kohala / northern tip) |

**Output:** `../data/DEM/hawaii_DEM_500m_utm.tif`  
Resolution: 500 m, CRS: EPSG:32604, resampling: average

In [1]:
import os
import subprocess

DEM_DIR = '../data/DEM'
os.makedirs(DEM_DIR, exist_ok=True)

## 1 — Download missing USGS 3DEP tiles

Tile URLs discovered via the USGS TNM API:
```
https://tnmaccess.nationalmap.gov/api/v1/products
    ?datasets=National Elevation Dataset (NED) 1/3 arc-second
    &bbox=-155.001,19.0,-154.999,19.001
```
The `/historical/` path and datestamp suffix in filenames must be fetched from
the API — the simple `StagedProducts` URL returns 404 for these tiles.

In [2]:
TILES = {
    # tile name       download URL (verified 2026-04-01)
    'USGS_13_n20w155_20250611.tif': (
        'https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/'
        'historical/n20w155/USGS_13_n20w155_20250611.tif'
    ),
    'USGS_13_n19w156_20230522.tif': (
        'https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/'
        'historical/n19w156/USGS_13_n19w156_20230522.tif'
    ),
    'USGS_13_n21w156_20230522.tif': (
        'https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/'
        'historical/n21w156/USGS_13_n21w156_20230522.tif'
    ),
}

for fname, url in TILES.items():
    dest = os.path.join(DEM_DIR, fname)
    if os.path.exists(dest):
        print(f'Already exists: {fname}')
        continue
    print(f'Downloading {fname} ...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', dest, url], check=True)
    size_mb = os.path.getsize(dest) / 1024**2
    print(f'  -> {size_mb:.0f} MB')

print('Done.')

Already exists: USGS_13_n20w155_20250611.tif
Already exists: USGS_13_n19w156_20230522.tif
Already exists: USGS_13_n21w156_20230522.tif
Done.


## 2 — Build island-wide 500m mosaic

`gdalwarp` with `-r average` reprojects all three tiles to UTM Zone 4N
(EPSG:32604) and downsamples to 500m in one pass.  Average resampling
excludes nodata cells, so coastal edge cells average only over valid land
pixels — equivalent to the `nanmean` block-averaging approach.

The output is DEFLATE-compressed; at 500m it is tiny (~149 KB).

In [3]:
# Source files — order matters: later files fill gaps in earlier ones
SOURCES = [
    os.path.join(DEM_DIR, 'kona_DEM_utm_full.tif'),          # 19-20N 155-156W (UTM)
    os.path.join(DEM_DIR, 'USGS_13_n20w155_20250611.tif'),   # 19-20N 154-155W (NAD83)
    os.path.join(DEM_DIR, 'USGS_13_n19w156_20230522.tif'),   # 18-19N 155-156W (NAD83)
    os.path.join(DEM_DIR, 'USGS_13_n21w156_20230522.tif'),   # 20-21N 155-156W (NAD83) — North Kohala
]

OUTPUT = os.path.join(DEM_DIR, 'hawaii_DEM_500m_utm.tif')

# Big Island bounding box (lat/lon) — clips out-of-extent tile coverage
# Northern tip (North Kohala) is ~20.27N; south is Ka Lae at ~18.91N
BIG_ISLAND_TE = ['-156.1', '18.85', '-154.78', '20.35']

if os.path.exists(OUTPUT):
    print(f'Output already exists: {OUTPUT}')
    print('Delete it first to regenerate.')
else:
    cmd = [
        'gdalwarp',
        '-t_srs', 'EPSG:32604',        # UTM Zone 4N
        '-tr', '500', '500',            # 500m output resolution
        '-r', 'average',                # average resampling (nanmean-equivalent)
        '-te_srs', 'EPSG:4326',         # interpret -te in lat/lon
        '-te', *BIG_ISLAND_TE,          # clip to Big Island extent
        '-dstnodata', '-9999',
        '-co', 'COMPRESS=DEFLATE',
    ] + SOURCES + [OUTPUT]

    print('Running gdalwarp ...')
    subprocess.run(cmd, check=True)
    size_kb = os.path.getsize(OUTPUT) / 1024
    print(f'Created {OUTPUT}: {size_kb:.0f} KB')

Output already exists: ../data/DEM/hawaii_DEM_500m_utm.tif
Delete it first to regenerate.


## 3 — Verify output

In [4]:
import rasterio
import numpy as np

with rasterio.open(OUTPUT) as src:
    dem = src.read(1).astype(np.float32)
    nd  = src.nodata
    print(f'Shape:       {src.height} rows x {src.width} cols')
    print(f'CRS:         {src.crs}')
    print(f'Resolution:  {src.res[0]:.0f} m')
    print(f'Bounds:      {src.bounds}')
    print(f'Nodata:      {nd}')

dem[dem == nd] = np.nan
dem[dem <= 0]  = np.nan   # mask ocean
land = ~np.isnan(dem)

print(f'Land cells:  {land.sum():,} of {dem.size:,} total')
print(f'Elev range:  {np.nanmin(dem):.0f} – {np.nanmax(dem):.0f} m')

Shape:       338 rows x 270 cols
CRS:         EPSG:32604
Resolution:  500 m
Bounds:      BoundingBox(left=805601.0921646439, bottom=2086864.7610769686, right=940601.0921646439, top=2255864.7610769686)
Nodata:      -9999.0
Land cells:  42,524 of 91,260 total
Elev range:  0 – 4166 m
